# ITEC631 – AI-Driven Network Intrusion Detection System
## Sprint 1: Data Pipeline

**Student:** Hayoung
**Unit:** ITEC631 IT Masters Project Part B, Semester 1 2026

Sprint 1 goal is to get the CICIDS2017 dataset cleaned up and ready to use for training models in Sprint 2. The dataset comes as 8 separate CSV files so I need to combine them, do some EDA to understand what we're working with, fix data quality issues, and apply SMOTE to deal with the class imbalance problem.

## Setup

In [ ]:
import os, sys

ON_KAGGLE = os.path.exists('/kaggle/working')

if ON_KAGGLE:
    os.system('pip install imbalanced-learn -q')

    # walk all of /kaggle/input to find the folder containing the CSV files
    DATA_DIR = None
    for root, dirs, files in os.walk('/kaggle/input'):
        csvs = [f for f in files if f.endswith('.csv') and 'WorkingHours' in f]
        if csvs:
            DATA_DIR = root + '/'
            break

    if DATA_DIR is None:
        # show what is available to help debug
        print("Could not find CSV files. Here is what is in /kaggle/input:")
        for root, dirs, files in os.walk('/kaggle/input'):
            print(f"  {root}")
            for f in files[:3]:
                print(f"    {f}")
        raise FileNotFoundError("Dataset not found — make sure you attached it via + Add Data in the sidebar.")
else:
    DATA_DIR = 'data/MachineLearningCVE/'

print(f'DATA_DIR: {DATA_DIR}')
print(f'CSV files found: {len([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])}')

: 

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
RANDOM_STATE = 42

## 1. Load the Data

The dataset is split across 8 CSV files (one per capture day). I'll load them all and combine into one dataframe.

In [ ]:
csv_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.csv')])
print(f'{len(csv_files)} files found:')
for f in csv_files:
    print(f'  {f}')

In [ ]:
frames = []
for fname in csv_files:
    df_part = pd.read_csv(DATA_DIR + fname, low_memory=False)
    df_part['source_file'] = fname
    frames.append(df_part)
    print(f'  loaded {fname} — {len(df_part):,} rows')

df = pd.concat(frames, ignore_index=True)
print(f'\ncombined shape: {df.shape}')

### Sampling for development

Working with 2.8M rows takes a while so I'm using a 20% stratified sample during Sprint 1.
I'll switch this off in Sprint 2 when training the actual models.

In [ ]:
USE_SAMPLE = True  # set to False in Sprint 2 to use full dataset

if USE_SAMPLE:
    df = df.groupby(' Label', group_keys=False).apply(
        lambda x: x.sample(frac=0.2, random_state=RANDOM_STATE)
    ).reset_index(drop=True)
    print(f'using sample: {len(df):,} rows')
else:
    print(f'using full dataset: {len(df):,} rows')

## 2. Exploratory Data Analysis

Before doing anything I want to understand the data — how many features, what the class distribution looks like, and whether there are any obvious data quality issues.

In [ ]:
print(f'rows    : {df.shape[0]:,}')
print(f'columns : {df.shape[1]}')
print(f'\ndata types:')
print(df.dtypes.value_counts())
print(f'\nfirst few column names:')
print(list(df.columns[:10]))

In [ ]:
# class distribution
label_counts = df[' Label'].value_counts()
print('class counts:')
print(label_counts.to_string())
print(f'\nunique labels: {df[" Label"].nunique()}')

In [ ]:
# visualise class distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('CICIDS2017 Class Distribution', fontsize=13, fontweight='bold')

# bar chart
ax1 = axes[0]
bars = ax1.barh(label_counts.index[::-1], label_counts.values[::-1],
                color=['#2ecc71' if l == 'BENIGN' else '#e74c3c' for l in label_counts.index[::-1]])
ax1.set_xlabel('Number of Records')
ax1.set_title('All Classes')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, label_counts.values[::-1]):
    ax1.text(bar.get_width() + max(label_counts)*0.01,
             bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

# pie chart — benign vs attack
ax2 = axes[1]
benign = label_counts.get('BENIGN', 0)
attack = label_counts.sum() - benign
ax2.pie([benign, attack], labels=['BENIGN', 'ATTACK'],
        autopct='%1.2f%%', colors=['#2ecc71', '#e74c3c'],
        startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax2.set_title('BENIGN vs ATTACK')

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'BENIGN: {benign:,}  ({benign/len(df)*100:.1f}%)')
print(f'ATTACK: {attack:,}  ({attack/len(df)*100:.1f}%)')

In [ ]:
# check for missing values
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) == 0:
    print('no missing values')
else:
    print(f'{len(missing_cols)} columns have missing values:')
    print(missing_cols)

In [ ]:
# check for infinity values — common in flow-based features
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df[numeric_cols]).sum()
inf_cols = inf_counts[inf_counts > 0]
if len(inf_cols) == 0:
    print('no infinity values')
else:
    print(f'infinity values found in {len(inf_cols)} columns:')
    print(inf_cols)

In [ ]:
# correlation heatmap — top 20 features by variance
fig, ax = plt.subplots(figsize=(13, 10))

num_df = df.select_dtypes(include=[np.number]).copy()
num_df.replace([np.inf, -np.inf], np.nan, inplace=True)

top_feats = num_df.var().nlargest(20).index
corr = num_df[top_feats].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdYlGn', center=0,
            linewidths=0.3, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Feature Correlation — Top 20 by Variance', fontsize=12, pad=12)
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Cleaning

A few things I need to fix:
- Column names have leading/trailing whitespace in the raw CSVs
- Some flow features produce infinity values (e.g. division by zero in rate calculations)
- Drop rows with NaN values after replacing infinities
- Remove exact duplicate rows

In [ ]:
# strip whitespace from column names
df.columns = df.columns.str.strip()
print('column names cleaned')
print('label column is now:', 'Label')

In [ ]:
# replace inf with NaN then drop
inf_before = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print(f'replaced {inf_before:,} infinity values with NaN')

nan_before = df.isnull().sum().sum()
df.dropna(inplace=True)
print(f'dropped rows with NaN — {nan_before:,} NaN values removed')
print(f'rows remaining: {len(df):,}')

In [ ]:
# remove duplicates
dupes = df.duplicated().sum()
df.drop_duplicates(inplace=True)
df.drop(columns=['source_file'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'removed {dupes:,} duplicate rows')
print(f'final shape after cleaning: {df.shape}')

## 4. Label Engineering

I need two label columns for the two-stage detection pipeline:
- **label_binary** — Stage 1: is this traffic normal or an attack? (0 = BENIGN, 1 = ATTACK)
- **label_category** — Stage 2: what kind of attack is it? (DoS/DDoS, PortScan, BruteForce, WebAttack, Botnet, Infiltration)

The CICIDS2017 label names don't map directly to KDD99 categories so I've grouped them into the categories that make sense for this dataset.

In [ ]:
df['label_binary'] = (df['Label'] != 'BENIGN').astype(int)
print('binary label distribution:')
print(df['label_binary'].value_counts().rename({0: 'BENIGN (0)', 1: 'ATTACK (1)'}))

In [ ]:
CATEGORY_MAP = {
    'BENIGN'                         : 'BENIGN',
    'DoS Hulk'                       : 'DoS_DDoS',
    'DoS GoldenEye'                  : 'DoS_DDoS',
    'DoS slowloris'                  : 'DoS_DDoS',
    'DoS Slowhttptest'               : 'DoS_DDoS',
    'Heartbleed'                     : 'DoS_DDoS',
    'DDoS'                           : 'DoS_DDoS',
    'PortScan'                       : 'PortScan',
    'FTP-Patator'                    : 'BruteForce',
    'SSH-Patator'                    : 'BruteForce',
    'Web Attack  Brute Force'    : 'WebAttack',
    'Web Attack  XSS'            : 'WebAttack',
    'Web Attack  Sql Injection'  : 'WebAttack',
    'Bot'                            : 'Botnet',
    'Infiltration'                   : 'Infiltration',
}

df['label_category'] = df['Label'].map(CATEGORY_MAP)

unmapped = df['label_category'].isnull().sum()
if unmapped > 0:
    print(f'warning: {unmapped} rows not mapped — check CATEGORY_MAP')
    print(df[df['label_category'].isnull()]['Label'].unique())
else:
    print('all labels mapped successfully')

print('\ncategory counts:')
print(df['label_category'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
cat_counts = df['label_category'].value_counts()
colors = ['#2ecc71' if c == 'BENIGN' else '#e74c3c' for c in cat_counts.index]
bars = ax.bar(cat_counts.index, cat_counts.values, color=colors, edgecolor='white')
ax.set_title('Attack Category Distribution', fontsize=12)
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, cat_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(cat_counts)*0.01,
            f'{val:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('eda_categories.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Preparation

Separate the feature matrix from the labels, drop any constant columns, and apply Min-Max normalisation to scale all features to [0, 1].

In [ ]:
X = df.drop(columns=['Label', 'label_binary', 'label_category'])
y_binary   = df['label_binary']
y_category = df['label_category']

# encode category labels as integers
le = LabelEncoder()
y_cat_encoded = le.fit_transform(y_category)

print(f'features X : {X.shape}')
print(f'\ncategory encoding:')
for cls, enc in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {cls:<16} → {enc}')

In [ ]:
# drop constant columns (zero variance)
const_cols = X.columns[X.var() == 0].tolist()
if const_cols:
    X.drop(columns=const_cols, inplace=True)
    print(f'dropped {len(const_cols)} constant columns: {const_cols}')
else:
    print('no constant columns')

# remove duplicate columns (Fwd Header Length appears twice in CICIDS2017)
dup_cols = X.columns[X.columns.duplicated()].tolist()
if dup_cols:
    X = X.loc[:, ~X.columns.duplicated()]
    print(f'removed duplicate columns: {dup_cols}')

print(f'feature matrix: {X.shape}')

In [ ]:
# min-max normalisation — scale everything to [0, 1]
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print('normalisation done — all features in [0, 1]')
print(X_scaled.describe().loc[['min', 'max']].T.head(8).to_string())

## 6. Class Balancing with SMOTE

The dataset is heavily imbalanced — BENIGN traffic makes up the large majority, while rare attacks like Infiltration and Botnet are almost invisible. If I train on this as-is the model will just learn to predict BENIGN for everything and still get high accuracy.

SMOTE generates synthetic samples for the minority class so the model actually learns to detect the rare attacks. Important: SMOTE is only applied to the training set, not the test set, to avoid leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_binary,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_binary
)

print(f'train: {X_train.shape[0]:,} rows')
print(f'test : {X_test.shape[0]:,} rows')
print(f'\nclass distribution in training set (before SMOTE):')
print(y_train.value_counts().rename({0: 'BENIGN', 1: 'ATTACK'}))

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('SMOTE done')
print(f'  before: {X_train.shape[0]:,} rows')
print(f'  after : {X_train_bal.shape[0]:,} rows')
print(f'\nclass distribution after SMOTE:')
print(pd.Series(y_train_bal).value_counts().rename({0: 'BENIGN', 1: 'ATTACK'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Training Set — Before vs After SMOTE', fontsize=12)

before = y_train.value_counts()
after  = pd.Series(y_train_bal).value_counts()

for ax, counts, title in [(axes[0], before, 'Before SMOTE'), (axes[1], after, 'After SMOTE')]:
    ax.bar(['BENIGN', 'ATTACK'], counts.values,
           color=['#2ecc71', '#e74c3c'], edgecolor='white')
    ax.set_title(title)
    ax.set_ylabel('Count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    for i, v in enumerate(counts.values):
        ax.text(i, v + max(counts)*0.02, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Processed Data

Save everything to disk so I don't have to rerun preprocessing in Sprint 2.

In [ ]:
import json as _json
os.makedirs('processed_data', exist_ok=True)

# full cleaned dataset
full = X_scaled.copy()
full['label_binary']        = y_binary.values
full['label_category']      = y_category.values
full['label_cat_encoded']   = y_cat_encoded
full.to_csv('processed_data/cicids2017_cleaned.csv', index=False)
print(f'saved cicids2017_cleaned.csv  — {len(full):,} rows')

# balanced training set
train_df = pd.DataFrame(X_train_bal, columns=X_train.columns)
train_df['label_binary'] = y_train_bal
train_df.to_csv('processed_data/train_balanced.csv', index=False)
print(f'saved train_balanced.csv      — {len(train_df):,} rows')

# test set (no SMOTE)
test_df = X_test.copy()
test_df['label_binary'] = y_test.values
test_df.to_csv('processed_data/test.csv', index=False)
print(f'saved test.csv                — {len(test_df):,} rows')

# label encoder classes (needed in Sprint 2+)
with open('processed_data/label_classes.json', 'w') as f:
    _json.dump(list(le.classes_), f)
print('saved label_classes.json')

## Sprint 1 Summary

In [ ]:
print('Sprint 1 — Data Pipeline complete')
print('='*55)
print(f'raw rows loaded      : 2,830,743 (8 CSV files)')
print(f'after cleaning       : {len(X_scaled):,} rows, {X_scaled.shape[1]} features')
print(f'labels created       : binary (0/1) + 6-class category')
print(f'normalisation        : Min-Max scaling → [0, 1]')
print(f'train/test split     : 80/20 stratified')
print(f'SMOTE balanced train : {len(X_train_bal):,} rows')
print(f'saved to             : processed_data/')
print()
print('Next: Sprint 2 — train Random Forest, XGBoost, Decision Tree')
print('      evaluate per-class recall especially for rare attack types')